In [11]:
!pip install scikit-learn pandas numpy matplotlib seaborn -q

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as snsd
import warnings
import nltk
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import PCA

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

warnings.filterwarnings("ignore", category=FutureWarning)

In [18]:
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
stop_words = joblib.load('stop_words.joblib')

In [7]:
df = pd.read_csv('unlabeled_df.csv')
df.shape

(12418, 2)

In [8]:
df = df.dropna(subset=['content'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12395 entries, 0 to 12417
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   content  12395 non-null  object 
 1   topic    0 non-null      float64
dtypes: float64(1), object(1)
memory usage: 290.5+ KB


In [12]:
def clean_text(text):

    text = str(text).lower() # нижний регистр

    text = re.sub(r'<[^>]+>', '', text) # чистим от html-тегов
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # чистим от url
    text = re.sub(r'[^а-яё\s]', '', text) # убираем знаки и цифры
    text = re.sub(r'\s+', ' ', text).strip() # убираем лишние пробелы

    return text

In [20]:
def preprocess_text(text, morph, stop_words):
    """
    Полная предобработка текста: очистка -> токенизация -> лемматизация -> фильтрация
    """

    text = clean_text(text)

    tokens = word_tokenize(text)
    tokens = [morph.parse(token)[0].normal_form for token in tokens]
    tokens = [token for token in tokens if token not in stop_words and len(token) > 1]

    return ' '.join(tokens)

In [21]:
df['content'] = df['content'].apply(clean_text)

,content
0,владимир зеленский фото марина совина пользова...
1,фото марина совина президент украины владимир ...
2,фото марина совина европейский союз ес призвал...
3,фото никита савин инопланетян редко изображают...
4,кадр фильм место встречи изменить нельзя натал...


In [27]:
vectorizer = TfidfVectorizer(
    max_features=5000,           # максимальное количество признаков
    max_df=0.8,                  # игнорировать слова, встречающиеся более чем в 80% документов
    min_df=5,                    # игнорировать слова, встречающиеся менее чем в 5 документах
    ngram_range=(1, 2)         # использовать униграммы и биграммы
)

In [28]:
tfidf_matrix = vectorizer.fit_transform(df['content'])
print(f"Размер TF-IDF матрицы: {tfidf_matrix.shape}")

Размер TF-IDF матрицы: (12395, 5000)
